# Classical control v3 — pre-registered robustness experiments (E1–E5), no neural network
**Purpose:** harden the v2.1 findings before rewriting the proposal. Criteria are frozen in `PREREG` (Cell 1; hash printed and saved). Anything changed after seeing results must be reported as *post hoc*.

| Exp | Question |
|---|---|
| E1 | Is free-field recovery (v2.1: r≈0.94 at the grid edge λ=10) real? λ grid extended to 1000; skill score instead of r alone |
| E2 | Is that r produced by the smoothness regulariser rather than by data? y-split truth, random-blob truth, and a NULL (uniform-Cf data) |
| E3 | Are the borderline zonal verdicts stable with 10 noise realisations and varied tolerances? |
| E4 | At which forcing U0 does the zonal model become identified? (U0 sweep) |
| E5 | What happens when the truth is outside the zonal class (curved boundary, 3 zones, gradient)? |

Not here: K-sweep with real fits (E6). Authoring status: the JAX-specific constructs and the core solver/problem/zonal code are reused from v2.1 (executed on Kaggle); the new v3 logic was exercised only with a NumPy mock of JAX at tiny settings. **Cell 3 (gate) must pass before anything is interpreted.** Estimated run time (estimate, not measurement): see RUNBOOK_V3.

In [1]:
import os, json, time, hashlib, math
import numpy as np
import jax, jax.numpy as jnp
from jax import lax
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter
jax.config.update("jax_enable_x64", True)

MODE = "quick"      # "quick" (pipeline check, minutes) | "standard"   [time estimates in the runbook are estimates, not measurements]
TEST_MODE = False      # authoring-sandbox smoke only; leave False on Kaggle
RUN = dict(E1=True, E2=True, E4=True, E3=True, E5=True)   # switch experiments off to run them in separate sessions
RESUME = True          # continue from v3_results.json if it exists in the output directory

# ====================================================================================
# PRE-REGISTERED CRITERIA  (frozen BEFORE any v3 run; the hash is printed and saved.)
# Anything changed after seeing results must be reported as POST HOC.
# ====================================================================================
PREREG = dict(
    version="v3-prereg-1",
    E1=dict(desc="free field, lambda grid extended upward; skill = 1 - RMSE/RMSE_constant (RMSE_constant = best constant predictor, oracle mean)",
            lambdas=[1000.0, 300.0, 100.0, 30.0, 10.0, 3.0, 1.0],
            recovery_if="skill >= 0.5 at the discrepancy-principle lambda (equivalently RMSE <= ~25% of Cf range)",
            interior_if="oracle (max-skill) lambda is not the largest or smallest grid value"),
    E2=dict(desc="regulariser-bias tests, h-only; (a) y-split truth (boundary roughly parallel to the flow), (b) random-blob truth, (c) NULL: data from uniform Cf",
            lambdas=[100.0, 30.0, 10.0, 3.0, 1.0],
            a_rule="r_DP(y-split) lower than r_DP(standard truth, E1, same config) by > 0.10 => 'not robust to geometry' (CONFOUNDED with physics: flow direction differs)",
            c_rule="NULL imprint established if r(Cf_hat, standard CF_T pattern) > 0.30 at DP lambda AND std(Cf_hat)/mean > 5%",
            note="a 90-degree rotation is not geometry-preserving on a 150x50 km domain, so (a) uses a y-split; (c) was ADDED to disentangle the confound"),
    E3=dict(desc="zonal 4-parameter, 10 noise realisations (layout+noise), rows = edge rows from v2.1",
            n_seeds=10, class_rule="valid seeds = identified + not_identified (optimiser_failed excluded); need >= 80% valid else INVALID; "
                                   "IDENTIFIED if identified >= ceil(0.9*valid); NOT_IDENTIFIED if identified <= floor(0.1*valid); else MARGINAL",
            primary="mean boundary error <= 6.25 km AND both Cf within 25% (as v2.1)",
            also_reported="Cf tolerance 15% / 35%; and edge-based boundary error (max displacement at y=0,1) <= 6.25 km; and counts excluding misspec-flagged seeds",
            misspec_flag="chi2_ref > 1 + 5*sqrt(2/N)"),
    E4=dict(desc="zonal h-only, U0 sweep, both regimes, same 10 seeds and class rule as E3",
            U0_list=[0.15, 0.3, 0.5, 0.7, 1.0],
            U0_star="smallest U0 whose class is IDENTIFIED with all larger U0 also IDENTIFIED; else reported as non-monotone / never"),
    E5=dict(desc="misspecification (no pass/fail): truths = curved boundary, three zones, continuous gradient; U0=1, h-only; zonal-4 vs free field; report r, RMSE, skill, chi2"),
    E6="NOT in this notebook (K-sweep with real fits postponed to next phase)",
)
PREREG_HASH = hashlib.sha256(json.dumps(PREREG, sort_keys=True).encode()).hexdigest()[:16]

_MODES = {
    "quick":    dict(N_STARTS=3, ZONAL_MAXITER=60,  FREE_MAXITER=120, N_SEEDS=2, N_SEEDS_E5=1,
                     L_E1=[100.0, 10.0, 1.0], L_E2=[30.0, 3.0], U0_E4=[0.15, 0.5, 1.0]),
    "standard": dict(N_STARTS=6, ZONAL_MAXITER=120, FREE_MAXITER=250, N_SEEDS=10, N_SEEDS_E5=3,
                     L_E1=PREREG["E1"]["lambdas"], L_E2=PREREG["E2"]["lambdas"], U0_E4=PREREG["E4"]["U0_list"]),
}
CFG = dict(
    N_STEPS=1500, T0=150, T_STRIDE=25, U0_LIST=[0.15, 1.0], P_OBS=39,
    NOISE_FRAC=0.03, SEED=7, CF_SCALE=2.5e-3,
    TOL_BND_KM=6.25, TOL_CF_REL=0.25, DCHI2_CRIT=13.3,
)
CFG.update(_MODES[MODE])
if TEST_MODE:
    CFG.update(N_STEPS=60, T0=20, T_STRIDE=10, N_STARTS=2, ZONAL_MAXITER=2, FREE_MAXITER=2, N_SEEDS=2, N_SEEDS_E5=1,
               L_E1=[10.0, 1.0], L_E2=[10.0], U0_E4=[0.5, 1.0], U0_LIST=[1.0])
REGIMES = {"rich": np.arange(CFG["T0"], CFG["N_STEPS"], CFG["T_STRIDE"]),
           "sparse4": np.linspace(CFG["N_STEPS"] // 4, CFG["N_STEPS"] - 1, 4).astype(int)}
if TEST_MODE: REGIMES = {"rich": REGIMES["rich"], "sparse4": REGIMES["sparse4"]}

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_PATH = os.path.join(OUT_DIR, "v3_results.json" if not TEST_MODE else "v3_results_TEST.json")
RESULTS = dict(mode=MODE, prereg=PREREG, prereg_hash=PREREG_HASH, cfg={k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in CFG.items()},
               E1=[], E2=[], zonal_rows={}, E3=[], E4=[], E5=[])
if RESUME and os.path.exists(OUT_PATH):
    _old = json.load(open(OUT_PATH))
    if _old.get("prereg_hash") == PREREG_HASH and _old.get("mode") == MODE:
        RESULTS.update({k: _old[k] for k in ("E1", "E2", "zonal_rows", "E3", "E4", "E5") if k in _old})
        print("RESUMED from", OUT_PATH)
    else:
        print("WARNING: existing results file has a different pre-registration hash or mode -> NOT resumed (kept untouched until first save).")

def _jsonable(o):
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
def save_results():
    with open(OUT_PATH, "w") as f: json.dump(RESULTS, f, default=_jsonable)

print("MODE:", MODE, "| devices:", jax.devices(), "| PRE-REGISTRATION HASH:", PREREG_HASH)
print({k: v for k, v in CFG.items()})
print({k: len(v) for k, v in REGIMES.items()}, "snapshots per regime")


MODE: quick | devices: [CpuDevice(id=0)] | PRE-REGISTRATION HASH: ab1b8ea0b153004f
{'N_STEPS': 1500, 'T0': 150, 'T_STRIDE': 25, 'U0_LIST': [0.15, 1.0], 'P_OBS': 39, 'NOISE_FRAC': 0.03, 'SEED': 7, 'CF_SCALE': 0.0025, 'TOL_BND_KM': 6.25, 'TOL_CF_REL': 0.25, 'DCHI2_CRIT': 13.3, 'N_STARTS': 3, 'ZONAL_MAXITER': 60, 'FREE_MAXITER': 120, 'N_SEEDS': 2, 'N_SEEDS_E5': 1, 'L_E1': [100.0, 10.0, 1.0], 'L_E2': [30.0, 3.0], 'U0_E4': [0.15, 0.5, 1.0]}
{'rich': 54, 'sparse4': 4} snapshots per regime


## Cell 2 — core physics + problem definition (unchanged from v2.1)

In [2]:
# ---- core physics + problem definition (xp-generic: numpy for truth/tests, jax.numpy for inversion) ----
LX, LY = 150e3, 50e3
NX, NY = 24, 12
DX, DY = LX / NX, LY / NY
G, H0, F0, DT = 9.81, 80.0, 6.5e-5, 60.0
OMEGA = 2 * np.pi / (12.42 * 3600.0)
CF_LOW_TRUE, CF_HIGH_TRUE = 1.5e-3, 4.0e-3
B0_TRUE, S_TRUE = 0.5, 0.1          # boundary x/LX = B0 + S*(y/LY - 0.5)  (identical to the phase-0 twin)
XC = (np.arange(NX) + 0.5) * DX / LX     # normalized cell-centre coordinates
YC = (np.arange(NY) + 0.5) * DY / LY

def swe_step(xp, state, t, Cf, U0):
    eta, u, v = state
    vp = xp.pad(v, ((1, 1), (0, 0)), mode="edge"); vc = 0.5 * (vp[:, :-1] + vp[:, 1:]); v_at_u = 0.5 * (vc[:-1, :] + vc[1:, :])
    uc = 0.5 * (u[:-1, :] + u[1:, :]); up = xp.pad(uc, ((0, 0), (1, 1)), mode="edge"); u_at_v = 0.5 * (up[:, :-1] + up[:, 1:])
    Cf_u = Cf[np.clip(np.arange(NX + 1), 0, NX - 1), :]
    Cf_v = Cf[:, np.clip(np.arange(NY + 1), 0, NY - 1)]
    su = xp.sqrt(u ** 2 + v_at_u ** 2 + 1e-8); sv = xp.sqrt(u_at_v ** 2 + v ** 2 + 1e-8)
    de_dx = (eta[1:, :] - eta[:-1, :]) / DX; de_dy = (eta[:, 1:] - eta[:, :-1]) / DY
    fu = Cf_u / H0 * u * su; fv = Cf_v / H0 * v * sv
    un = u[1:-1, :] + DT * (-G * de_dx + F0 * v_at_u[1:-1, :] - fu[1:-1, :])
    u_new = xp.concatenate([xp.full((1, NY), U0 * xp.cos(OMEGA * t)), un, xp.zeros((1, NY))], axis=0)
    vn = v[:, 1:-1] + DT * (-G * de_dy - F0 * u_at_v[:, 1:-1] - fv[:, 1:-1])
    v_new = xp.concatenate([xp.zeros((NX, 1)), vn, xp.zeros((NX, 1))], axis=1)
    div = (u_new[1:, :] - u_new[:-1, :]) / DX + (v_new[:, 1:] - v_new[:, :-1]) / DY
    return (eta - DT * H0 * div, u_new, v_new)

def run_obs_np(Cf, U0, n_steps, ox, oy):
    """numpy reference run -> (h,u,v) at observed cells for every step: 3 arrays (n_steps, P)."""
    s = (np.zeros((NX, NY)), np.zeros((NX + 1, NY)), np.zeros((NX, NY + 1)))
    H, U, V = [], [], []
    for n in range(n_steps):
        s = swe_step(np, s, n * DT, Cf, U0)
        e, u, v = s
        H.append(e[ox, oy]); U.append((0.5 * (u[:-1] + u[1:]))[ox, oy]); V.append((0.5 * (v[:, :-1] + v[:, 1:]))[ox, oy])
    return np.array(H), np.array(U), np.array(V)

def cf_zonal(xp, p, w_cells=0.5):
    """p = [ln Cf_low, ln Cf_high, b0, s]; smooth two-zone field on the cell grid (width = w_cells cells)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    arg = (X - (p[2] + p[3] * (Y - 0.5))) / (w_cells * DX / LX)
    sig = 1.0 / (1.0 + xp.exp(-arg))
    lo, hi = xp.exp(p[0]), xp.exp(p[1])
    return lo + (hi - lo) * sig

def cf_true_field():
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    return np.where(X < B0_TRUE + S_TRUE * (Y - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE)


# ---- K-zone model: K strips along x separated by K-1 parallel (common-slope) smooth boundaries ----
def cf_zonal_K(xp, theta, K, w_cells=0.5):
    """theta = [ln c_1..ln c_K, b_1..b_{K-1}, s]  (K=1: [ln c_1]).  Boundary k: x/LX = b_k + s*(y/LY-0.5)."""
    X, Y = np.meshgrid(XC, YC, indexing="ij")
    f = xp.exp(theta[0]) * xp.ones((NX, NY))
    for k in range(K - 1):
        arg = (X - (theta[K + k] + theta[2 * K - 1] * (Y - 0.5))) / (w_cells * DX / LX)
        f = f + (xp.exp(theta[k + 1]) - xp.exp(theta[k])) / (1.0 + xp.exp(-arg))
    return f

# ---- JAX forward model returning only the observed cells (keeps memory tiny) ----
def make_forward(U0, ox, oy):
    oxj, oyj = jnp.array(ox), jnp.array(oy)
    def fwd(Cf):
        def body(state, n):
            new = swe_step(jnp, state, n * DT, Cf, U0)
            e, u, v = new
            uc = 0.5 * (u[:-1, :] + u[1:, :]); vc = 0.5 * (v[:, :-1] + v[:, 1:])
            return new, (e[oxj, oyj], uc[oxj, oyj], vc[oxj, oyj])
        init = (jnp.zeros((NX, NY)), jnp.zeros((NX + 1, NY)), jnp.zeros((NX, NY + 1)))
        _, ys = lax.scan(body, init, jnp.arange(CFG["N_STEPS"]))
        return ys
    return fwd

CF_T = cf_true_field()
BOUNDS = [(np.log(5e-4), np.log(2e-2))] * 2 + [(0.1, 0.9), (-0.5, 0.5)]
P_TRUE = np.array([np.log(CF_LOW_TRUE), np.log(CF_HIGH_TRUE), B0_TRUE, S_TRUE])

def build_problem(U0, regime, variant, P_use, noise_seed=0, cf_truth=None):
    """Truth from the independent NumPy loop; inversion uses the JAX solver (same equations => 'inverse crime',
    acceptable for an identifiability test, NOT for validation). Layout and noise change with noise_seed."""
    cf_truth = CF_T if cf_truth is None else cf_truth
    rng = np.random.default_rng(CFG["SEED"] + noise_seed)
    cells = rng.choice(NX * NY, CFG["P_OBS"], replace=False)
    ox, oy = np.unravel_index(cells[:P_use], (NX, NY))
    t_idx = REGIMES[regime]
    Hc, Uc, Vc = run_obs_np(cf_truth, U0, CFG["N_STEPS"], ox, oy)
    clean = dict(h=Hc[t_idx], u=Uc[t_idx], v=Vc[t_idx])
    sd = {k: CFG["NOISE_FRAC"] * clean[k].std() for k in clean}
    nr = np.random.default_rng(11 + noise_seed)
    obs = {k: clean[k] + nr.normal(scale=sd[k], size=clean[k].shape) for k in clean}
    fwd = make_forward(U0, ox, oy)
    tj = jnp.array(t_idx)
    def chi2(Cf):
        H, U, V = fwd(Cf)
        c = jnp.mean(((H[tj] - obs["h"]) / sd["h"]) ** 2)
        if variant == "huv":
            c = (c + jnp.mean(((U[tj] - obs["u"]) / sd["u"]) ** 2) + jnp.mean(((V[tj] - obs["v"]) / sd["v"]) ** 2)) / 3.0
        return c
    return dict(U0=U0, regime=regime, variant=variant, P=P_use, ox=ox, oy=oy, t_idx=t_idx, obs=obs, sd=sd,
                fwd=fwd, chi2=chi2, n_obs=int(obs["h"].size * (3 if variant == "huv" else 1)))

## Cell 2b — alternative truth fields (E2, E5) and the NULL truth

In [3]:
# ---------------- truth fields (E2, E5) ----------------
_X, _Y = np.meshgrid(XC, YC, indexing="ij")
_MID = 0.5 * (CF_LOW_TRUE + CF_HIGH_TRUE)
TRUTHS = {
    "standard":   CF_T,
    "ysplit":     np.where(_Y < 0.5 + 0.1 * (_X - 0.5), CF_LOW_TRUE, CF_HIGH_TRUE),                       # boundary ~parallel to the flow
    "blobs":      np.where((lambda z: z > np.median(z))(gaussian_filter(np.random.default_rng(2024).normal(size=(NX, NY)), 2.0, mode="reflect")),
                           CF_HIGH_TRUE, CF_LOW_TRUE),
    "curved":     np.where(_X < 0.5 + 0.1 * (_Y - 0.5) + 0.08 * np.sin(2 * np.pi * _Y), CF_LOW_TRUE, CF_HIGH_TRUE),   # +-12 km wiggle
    "three_zone": np.where(_X < 0.35 + 0.1 * (_Y - 0.5), CF_LOW_TRUE, np.where(_X < 0.65 + 0.1 * (_Y - 0.5), _MID, CF_HIGH_TRUE)),
    "gradient":   CF_LOW_TRUE + (CF_HIGH_TRUE - CF_LOW_TRUE) * _X,                                          # continuous
    "uniform":    np.full((NX, NY), 2.5e-3),                                                                # NULL truth for E2(c)
}
def _corr(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()
    if a.std() < 1e-12 or b.std() < 1e-12: return 0.0
    return float(np.corrcoef(a, b)[0, 1])
print("truth-pattern correlations with the standard truth:",
      {k: round(_corr(v, CF_T), 2) for k, v in TRUTHS.items() if k not in ("standard", "uniform")})



truth-pattern correlations with the standard truth: {'ysplit': -0.08, 'blobs': 0.08, 'curved': 0.92, 'three_zone': 0.83, 'gradient': 0.86}


## Cell 3 — GATE (v2.1 checks + every truth field must give χ²≈1 at its own parameters)

In [4]:
gate = {}
pb = build_problem(CFG["U0_LIST"][-1], "rich", "h", CFG["P_OBS"])
H_j, U_j, V_j = [np.asarray(a) for a in pb["fwd"](jnp.array(CF_T))]
H_n, U_n, V_n = run_obs_np(CF_T, pb["U0"], CFG["N_STEPS"], pb["ox"], pb["oy"])
rel = np.max(np.abs(H_j - H_n)) / np.std(H_n)
print(f"(a) JAX vs NumPy solver: max|dh|/std(h) = {rel:.2e}  ->", "PASS" if rel < 1e-8 else "FAIL"); gate["solver_match"] = bool(rel < 1e-8)
loss_p = lambda p: pb["chi2"](cf_zonal(jnp, p))
vg = jax.jit(jax.value_and_grad(loss_p))
p0 = P_TRUE + np.array([0.2, -0.2, 0.03, -0.02])
v0, g0 = vg(jnp.array(p0)); g0 = np.asarray(g0)
fd = np.zeros(4)
for i in range(4):
    e = np.zeros(4); e[i] = 1e-6
    fd[i] = (float(loss_p(jnp.array(p0 + e))) - float(loss_p(jnp.array(p0 - e)))) / 2e-6
err = np.max(np.abs(g0 - fd) / (np.abs(fd) + 1e-12))
print(f"(b) grad check: max rel err = {err:.2e} ->", "PASS" if err < 1e-3 else "FAIL"); gate["grad_ok"] = bool(err < 1e-3)
c_ref = float(loss_p(jnp.array(P_TRUE)))
print(f"(c) chi2 at true params (h-only, U0={pb['U0']}): {c_ref:.3f} (expect ~1.0 +/- {2*np.sqrt(2/pb['n_obs']):.3f})")
gate["chi2_ok"] = bool(abs(c_ref - 1) < 4 * np.sqrt(2 / pb["n_obs"]))
gcf = np.asarray(jax.grad(pb["chi2"])(jnp.array(CF_T)))
print(f"(d) informational: d chi2 / d Cf[0,0] = {gcf[0,0]:.1e} (structural null cell: never enters the solver stencil); max|d chi2/d Cf| = {np.abs(gcf).max():.1e}")
assert all(gate.values()), "GATE FAILED — do not interpret anything below; send this cell's output to the assistant."
# ---- v3 additions to the gate: every truth field must reproduce chi2 ~ 1 at its own parameters (solver + noise model consistent) ----
_Uref = CFG["U0_LIST"][-1]
for _name, _T in TRUTHS.items():
    _pbt = build_problem(_Uref, "rich", "h", CFG["P_OBS"], cf_truth=_T)
    _c = float(_pbt["chi2"](jnp.array(_T)))
    _ok = bool(np.isfinite(_c) and abs(_c - 1) < 4 * np.sqrt(2 / _pbt["n_obs"]))
    print(f"(e) truth '{_name:10s}': chi2 at its own field = {_c:.3f} -> ", "PASS" if _ok else "FAIL"); gate["truth_" + _name] = _ok
assert all(gate.values()), "GATE (v3 additions) FAILED — do not interpret anything below; send this cell's output to the assistant."
print("GATE: all checks passed" if all(gate.values()) else "GATE FAILED")


(a) JAX vs NumPy solver: max|dh|/std(h) = 8.07e-15  -> PASS
(b) grad check: max rel err = 1.91e-09 -> PASS
(c) chi2 at true params (h-only, U0=1.0): 0.999 (expect ~1.0 +/- 0.062)
(d) informational: d chi2 / d Cf[0,0] = 0.0e+00 (structural null cell: never enters the solver stencil); max|d chi2/d Cf| = 6.1e-01
(e) truth 'standard  ': chi2 at its own field = 0.998 ->  PASS
(e) truth 'ysplit    ': chi2 at its own field = 0.998 ->  PASS
(e) truth 'blobs     ': chi2 at its own field = 0.998 ->  PASS
(e) truth 'curved    ': chi2 at its own field = 0.998 ->  PASS
(e) truth 'three_zone': chi2 at its own field = 0.998 ->  PASS
(e) truth 'gradient  ': chi2 at its own field = 0.998 ->  PASS
(e) truth 'uniform   ': chi2 at its own field = 0.998 ->  PASS
GATE: all checks passed


## Cell 4 — shared tools (truth fields, skill score, free-field path, zonal fit/classification, resumable cache)

In [5]:
def field_metrics(cf_hat, cf_ref):
    """r, RMSE (% of the reference range) and skill = 1 - RMSE/RMSE_constant (constant predictor = oracle mean of the reference)."""
    rng_ = float(cf_ref.max() - cf_ref.min())
    rmse = float(np.sqrt(np.mean((cf_hat - cf_ref) ** 2)) / rng_)
    rmse_c = float(np.sqrt(np.mean((cf_ref - cf_ref.mean()) ** 2)) / rng_)
    return dict(r=_corr(cf_hat, cf_ref), rmse_pct=100 * rmse, skill=(1 - rmse / rmse_c) if rmse_c > 0 else float("nan"))

# ---------------- free field: lambda continuation (as v2.1, + skill, + metric reference) ----------------
def fit_free_path(pb, lams, metric_ref):
    q = np.full(NX * NY, np.log(2.5e-3)); rows, fields = [], {}
    for lam in sorted(lams, reverse=True):
        def obj(qv, lam=lam):
            lnc = qv.reshape(NX, NY)
            smooth = jnp.mean((lnc[1:, :] - lnc[:-1, :]) ** 2) + jnp.mean((lnc[:, 1:] - lnc[:, :-1]) ** 2)
            return pb["chi2"](jnp.exp(lnc)) + lam * smooth
        vgf = jax.jit(jax.value_and_grad(obj))
        def f(qv):
            v, g = vgf(jnp.array(qv)); return float(v), np.asarray(g, dtype=np.float64)
        res = minimize(f, q, jac=True, method="L-BFGS-B", bounds=[(np.log(5e-4), np.log(2e-2))] * (NX * NY),
                       options=dict(maxiter=CFG["FREE_MAXITER"], ftol=1e-12, gtol=1e-8))
        q = res.x; cf = np.exp(q).reshape(NX, NY)
        m = field_metrics(cf, metric_ref)
        rows.append(dict(lam=float(lam), **m, chi2_data=float(pb["chi2"](jnp.array(cf))), cf_std=float(cf.std()), cf_mean=float(cf.mean()),
                         nit=int(res.nit), conv=bool(res.nit < CFG["FREE_MAXITER"])))
        fields[float(lam)] = cf
    tol = 1 + 2 * np.sqrt(2 / pb["n_obs"])
    ok = [i for i, r_ in enumerate(rows) if r_["chi2_data"] <= tol]
    dp = ok[0] if ok else len(rows) - 1                    # rows descend in lambda: first = largest lambda at noise level
    oracle = int(np.nanargmax([r_["skill"] if np.isfinite(r_["skill"]) else -9 for r_ in rows]))
    return rows, dp, oracle, fields

def print_free(tag, rows, dp, oracle):
    print(tag)
    for i, r_ in enumerate(rows):
        t = ("DP " if i == dp else "   ") + ("ORC" if i == oracle else "   ")
        print(f"        {t} lam={r_['lam']:7.1f}  r={r_['r']:6.3f}  RMSE={r_['rmse_pct']:6.1f}%  skill={r_['skill']:6.3f}  chi2={r_['chi2_data']:.3f}  std={r_['cf_std']:.5f}  nit={r_['nit']}  conv={r_['conv']}")

# ---------------- zonal 4-parameter (v2.1 fit + edge error) ----------------
def fit_zonal(pb, p0):
    vgf = jax.jit(jax.value_and_grad(lambda p: pb["chi2"](cf_zonal(jnp, p))))
    def f(p):
        v, g = vgf(jnp.array(p)); return float(v), np.asarray(g, dtype=np.float64)
    res = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=BOUNDS, options=dict(maxiter=CFG["ZONAL_MAXITER"], ftol=1e-12, gtol=1e-8))
    p = res.x
    off = float((p[2] - B0_TRUE) * LX / 1e3); tilt = float((p[3] - S_TRUE) * LX / 1e3)
    return dict(cf_low=float(np.exp(p[0])), cf_high=float(np.exp(p[1])), b0=float(p[2]), s=float(p[3]), chi2=float(res.fun), nit=int(res.nit),
                bnd_km=float(np.mean(np.abs((p[2] - B0_TRUE) + (p[3] - S_TRUE) * (YC - 0.5))) * LX / 1e3),
                off_km=off, tilt_km=tilt, edge_km=float(max(abs(off - tilt / 2), abs(off + tilt / 2))))

def zonal_one_seed(U0, regime, variant, P_use, ns, cf_truth=None):
    pbz = build_problem(U0, regime, variant, P_use, noise_seed=ns, cf_truth=cf_truth)
    chi2_ref = float(pbz["chi2"](cf_zonal(jnp, jnp.array(P_TRUE))))
    sols = []
    for k in range(CFG["N_STARTS"]):
        r = np.random.default_rng(100 + k + 17 * ns)
        p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
        sols.append(fit_zonal(pbz, p0))
    return dict(noise_seed=ns, chi2_ref=chi2_ref, n_obs=pbz["n_obs"], sols=sols)

def classify_seed(sd, tol_cf=0.25, tol_bnd=6.25, use_edge=False):
    n = sd["n_obs"]; sols = sd["sols"]; best = min(s["chi2"] for s in sols)
    if best > sd["chi2_ref"] + 2 * np.sqrt(2 / n): return "optimiser_failed"
    equiv = [s for s in sols if n * (s["chi2"] - best) <= CFG["DCHI2_CRIT"]]
    key = "edge_km" if use_edge else "bnd_km"
    ok = all(s[key] <= tol_bnd and abs(s["cf_low"] / CF_LOW_TRUE - 1) <= tol_cf and abs(s["cf_high"] / CF_HIGH_TRUE - 1) <= tol_cf for s in equiv)
    return "identified" if ok else "not_identified"

def misspec_flag(sd): return sd["chi2_ref"] > 1 + 5 * np.sqrt(2 / sd["n_obs"])

def classify_row(per_seed, tol_cf=0.25, tol_bnd=6.25, use_edge=False, exclude_flagged=False):
    seeds = [sd for sd in per_seed if not (exclude_flagged and misspec_flag(sd))]
    labs = [classify_seed(sd, tol_cf, tol_bnd, use_edge) for sd in seeds]
    ni, nn, nf = labs.count("identified"), labs.count("not_identified"), labs.count("optimiser_failed")
    valid = ni + nn
    if valid < 0.8 * len(seeds) or valid == 0: cls = "INVALID"
    elif ni >= math.ceil(0.9 * valid): cls = "IDENTIFIED"
    elif ni <= math.floor(0.1 * valid): cls = "NOT_IDENTIFIED"
    else: cls = "MARGINAL"
    return dict(identified=ni, not_identified=nn, optimiser_failed=nf, n_seeds=len(seeds), cls=cls)

def run_zonal_row(U0, regime, variant, n_seeds):
    key = f"U0={U0}|{regime}|{variant}"
    rows = RESULTS["zonal_rows"].setdefault(key, [])
    P_use = CFG["P_OBS"] if variant == "h" else CFG["P_OBS"] // 3
    t0 = time.time()
    for ns in range(len(rows), n_seeds):
        rows.append(zonal_one_seed(U0, regime, variant, P_use, ns)); save_results()
    c = classify_row(rows)
    print(f"[zonal] {key:28s} seeds={len(rows)}  identified {c['identified']}  not-id {c['not_identified']}  opt-failed {c['optimiser_failed']}  -> {c['cls']}   "
          f"misspec-flagged seeds: {sum(misspec_flag(s) for s in rows)}   [{time.time()-t0:.0f}s this call]")
    return rows


## Cell 5 — E1: free field, λ up to 1000, skill score
Recovery := skill ≥ 0.5 at the discrepancy-principle λ. 'Interior' := oracle λ is not at either grid edge.

In [6]:
# ======================= E1: free field, lambda up to 1000, skill score =======================
E1_CONFIGS = [(U0, reg, var) for U0 in CFG["U0_LIST"] for reg in REGIMES for var in ("h", "huv")]
done = {(r["U0"], r["regime"], r["variant"]) for r in RESULTS["E1"]}
t0 = time.time()
if RUN["E1"]:
    for (U0, reg, var) in E1_CONFIGS:
        if (U0, reg, var) in done: continue
        P_use = CFG["P_OBS"] if var == "h" else CFG["P_OBS"] // 3
        pb = build_problem(U0, reg, var, P_use, noise_seed=0)
        rows, dp, orc, fields = fit_free_path(pb, CFG["L_E1"], CF_T)
        RESULTS["E1"].append(dict(U0=U0, regime=reg, variant=var, noise_seed=0, n_obs=pb["n_obs"], rows=rows, dp=dp, oracle=orc,
                                  skill_dp=rows[dp]["skill"], skill_oracle=rows[orc]["skill"], r_dp=rows[dp]["r"],
                                  oracle_interior=bool(0 < orc < len(rows) - 1), recovery=bool(rows[dp]["skill"] >= 0.5),
                                  field_dp=fields[rows[dp]["lam"]].tolist(), field_oracle=fields[rows[orc]["lam"]].tolist()))
        save_results()
        print_free(f"[E1] U0={U0} {reg} {var} nobs={pb['n_obs']}  [{time.time()-t0:.0f}s]", rows, dp, orc)
print("\n=== E1 summary (pre-registered: recovery := skill >= 0.5 at DP lambda) ===")
print(f"{'U0':>5} {'regime':8s} {'var':4s} | {'lam_DP':>7} {'r_DP':>6} {'skill_DP':>8} {'recovery':>8} | {'lam_ORC':>7} {'skill_ORC':>9} {'ORC interior?':>13}")
for r in RESULTS["E1"]:
    print(f"{r['U0']:>5} {r['regime']:8s} {r['variant']:4s} | {r['rows'][r['dp']]['lam']:7.1f} {r['r_dp']:6.2f} {r['skill_dp']:8.2f} {str(r['recovery']):>8} | "
          f"{r['rows'][r['oracle']]['lam']:7.1f} {r['skill_oracle']:9.2f} {str(r['oracle_interior']):>13}")


[E1] U0=0.15 rich h nobs=2106  [37s]
        DP ORC lam=  100.0  r=-0.070  RMSE=  50.2%  skill=-0.003  chi2=0.996  std=0.00000  nit=108  conv=True
               lam=   10.0  r=-0.144  RMSE=  50.3%  skill=-0.006  chi2=0.995  std=0.00002  nit=68  conv=True
               lam=    1.0  r=-0.344  RMSE=  52.7%  skill=-0.054  chi2=0.994  std=0.00016  nit=63  conv=True
[E1] U0=0.15 rich huv nobs=2106  [89s]
        DP     lam=  100.0  r= 0.622  RMSE=  48.9%  skill= 0.021  chi2=1.010  std=0.00006  nit=94  conv=True
               lam=   10.0  r= 0.833  RMSE=  42.3%  skill= 0.153  chi2=0.996  std=0.00025  nit=112  conv=True
           ORC lam=    1.0  r= 0.898  RMSE=  30.2%  skill= 0.396  chi2=0.986  std=0.00061  nit=120  conv=False
[E1] U0=0.15 sparse4 h nobs=156  [127s]
        DP     lam=  100.0  r= 0.757  RMSE=  48.1%  skill= 0.038  chi2=0.885  std=0.00008  nit=90  conv=True
               lam=   10.0  r= 0.817  RMSE=  40.9%  skill= 0.183  chi2=0.872  std=0.00031  nit=79  conv=True
        

## Cell 6 — E2: regulariser-bias tests
(a) y-split truth (boundary roughly parallel to the flow; a 90° rotation is not geometry-preserving on 150×50 km, and the flow direction changes the physics → confounded). (b) random blobs. (c) NULL: data from uniform Cf; if the free-field estimate still correlates with the standard pattern, the regulariser/boundary conditions imprint structure.

In [7]:
# ======================= E2: regulariser-bias tests (h-only) =======================
# (a) y-split truth, (b) random blobs, (c) NULL: data from uniform Cf, correlated with the standard CF_T pattern.
E2_CONFIGS = [(U0, reg) for U0 in CFG["U0_LIST"] for reg in REGIMES]
done2 = {(r["U0"], r["regime"], r["truth"]) for r in RESULTS["E2"]}
t0 = time.time()
if RUN["E2"]:
    for (U0, reg) in E2_CONFIGS:
        for tname in ("ysplit", "blobs", "uniform"):
            if (U0, reg, tname) in done2: continue
            T = TRUTHS[tname]
            pb = build_problem(U0, reg, "h", CFG["P_OBS"], noise_seed=0, cf_truth=T)
            ref = CF_T if tname == "uniform" else T          # NULL: correlate with the *standard* pattern
            rows, dp, orc, _ = fit_free_path(pb, CFG["L_E2"], ref)
            RESULTS["E2"].append(dict(U0=U0, regime=reg, truth=tname, rows=rows, dp=dp, oracle=orc, r_dp=rows[dp]["r"], skill_dp=rows[dp]["skill"],
                                      cf_std_rel_dp=rows[dp]["cf_std"] / max(rows[dp]["cf_mean"], 1e-12)))
            save_results()
            print_free(f"[E2] U0={U0} {reg} truth={tname}  [{time.time()-t0:.0f}s]", rows, dp, orc)

print("\n=== E2 summary ===")
print("(a) y-split vs standard (E1, same config, h-only; r at DP lambda).  Rule: drop > 0.10 => 'not robust to geometry' (confounded with physics)")
for (U0, reg) in E2_CONFIGS:
    e1 = [r for r in RESULTS["E1"] if (r["U0"], r["regime"], r["variant"]) == (U0, reg, "h")]
    ys = [r for r in RESULTS["E2"] if (r["U0"], r["regime"], r["truth"]) == (U0, reg, "ysplit")]
    bl = [r for r in RESULTS["E2"] if (r["U0"], r["regime"], r["truth"]) == (U0, reg, "blobs")]
    if not (e1 and ys): print(f"  U0={U0} {reg}: missing E1 or E2 row (run E1 first)"); continue
    drop = e1[0]["r_dp"] - ys[0]["r_dp"]
    print(f"  U0={U0:<4} {reg:8s}: r_std={e1[0]['r_dp']:.2f}  r_ysplit={ys[0]['r_dp']:.2f}  drop={drop:+.2f} -> {'NOT ROBUST' if drop > 0.10 else 'ok'}"
          + (f" | blobs: r={bl[0]['r_dp']:.2f} skill={bl[0]['skill_dp']:.2f}" if bl else ""))
print("(c) NULL (uniform-Cf data): imprint if r(Cf_hat, CF_T) > 0.30 at DP lambda AND std/mean > 5%")
for r in [x for x in RESULTS["E2"] if x["truth"] == "uniform"]:
    imprint = (r["r_dp"] > 0.30) and (r["cf_std_rel_dp"] > 0.05)
    print(f"  U0={r['U0']:<4} {r['regime']:8s}: r_null(DP)={r['r_dp']:.2f}  std/mean={r['cf_std_rel_dp']:.3f}  -> {'IMPRINT' if imprint else 'no imprint'}")


[E2] U0=0.15 rich truth=ysplit  [27s]
        DP     lam=   30.0  r= 0.258  RMSE=  49.7%  skill= 0.006  chi2=1.001  std=0.00003  nit=90  conv=True
           ORC lam=    3.0  r= 0.297  RMSE=  48.1%  skill= 0.038  chi2=0.999  std=0.00023  nit=80  conv=True
[E2] U0=0.15 rich truth=blobs  [50s]
        DP ORC lam=   30.0  r= 0.207  RMSE=  50.2%  skill=-0.004  chi2=1.006  std=0.00006  nit=84  conv=True
               lam=    3.0  r= 0.121  RMSE=  51.5%  skill=-0.029  chi2=0.999  std=0.00046  nit=64  conv=True
[E2] U0=0.15 rich truth=uniform  [74s]
        DP ORC lam=   30.0  r=-0.840  RMSE=  51.6%  skill=-0.031  chi2=0.998  std=0.00002  nit=78  conv=True
               lam=    3.0  r=-0.859  RMSE=  57.6%  skill=-0.152  chi2=0.997  std=0.00020  nit=75  conv=True
[E2] U0=0.15 sparse4 truth=ysplit  [98s]
        DP     lam=   30.0  r= 0.642  RMSE=  47.3%  skill= 0.053  chi2=0.906  std=0.00011  nit=85  conv=True
           ORC lam=    3.0  r= 0.863  RMSE=  32.8%  skill= 0.344  chi2=0.880  std=

## Cell 7 — E4 (U0 sweep) and E3 (borderline rows, 10 seeds, tolerance variants)
One cache of zonal fits serves both. Resumable: re-running continues from `v3_results.json`.

In [8]:
# ======================= E4 (U0 sweep, h-only) and E3 (extra huv edge rows) share one cache of zonal fits =======================
t0 = time.time()
if RUN["E4"]:
    for reg in REGIMES:
        for U0 in CFG["U0_E4"]:
            run_zonal_row(U0, reg, "h", CFG["N_SEEDS"])
if RUN["E3"]:
    for (U0, reg) in [(0.15, "rich"), (1.0, "rich"), (1.0, "sparse4")]:
        if U0 in CFG["U0_LIST"]: run_zonal_row(U0, reg, "huv", CFG["N_SEEDS"])
    # the h-only edge rows of E3 ((1.0,rich),(1.0,sparse4)) are the U0=1.0 rows of E4 (same code, same seeds)
    for (U0, reg) in [(1.0, "rich"), (1.0, "sparse4")]:
        if RUN["E4"] is False and U0 in CFG["U0_LIST"]: run_zonal_row(U0, reg, "h", CFG["N_SEEDS"])

VARIANTS = [("PRIMARY: Cf 25%, mean-bnd", dict(tol_cf=0.25, use_edge=False)),
            ("Cf 15%, mean-bnd",         dict(tol_cf=0.15, use_edge=False)),
            ("Cf 35%, mean-bnd",         dict(tol_cf=0.35, use_edge=False)),
            ("Cf 25%, EDGE-bnd",         dict(tol_cf=0.25, use_edge=True)),
            ("Cf 25%, mean-bnd, excl. misspec-flagged", dict(tol_cf=0.25, use_edge=False, exclude_flagged=True))]
print("\n=== E3: zonal rows, class under pre-registered criteria and robustness variants (id/not-id/opt-fail -> class) ===")
RESULTS["E3"] = []
for key, per_seed in RESULTS["zonal_rows"].items():
    rec = dict(key=key, n_seeds=len(per_seed), variants={})
    print(f"{key}  (seeds={len(per_seed)}, misspec-flagged={sum(misspec_flag(s) for s in per_seed)})")
    for name, kw in VARIANTS:
        c = classify_row(per_seed, **kw); rec["variants"][name] = c
        print(f"      {name:42s} {c['identified']:2d}/{c['not_identified']:2d}/{c['optimiser_failed']:2d} -> {c['cls']}")
    RESULTS["E3"].append(rec)

print("\n=== E4: U0 sweep (h-only), PRIMARY class per U0; U0* = smallest U0 IDENTIFIED with all larger also IDENTIFIED ===")
RESULTS["E4"] = []
for reg in REGIMES:
    tab = []
    for U0 in sorted(CFG["U0_E4"]):
        rows_ = RESULTS["zonal_rows"].get(f"U0={U0}|{reg}|h")
        if rows_: tab.append((U0, classify_row(rows_)))
    ustar = None
    for i, (u, c) in enumerate(tab):
        if all(cc["cls"] == "IDENTIFIED" for _, cc in tab[i:]): ustar = u; break
    RESULTS["E4"].append(dict(regime=reg, table=[(u, c) for u, c in tab], U0_star=ustar))
    print(f"  {reg:8s}: " + "  ".join(f"U0={u}: {c['identified']}/{c['not_identified']}->{c['cls']}" for u, c in tab) + f"   => U0* = {ustar if ustar is not None else 'none (never / non-monotone)'}")
save_results()


[zonal] U0=0.15|rich|h               seeds=2  identified 1  not-id 1  opt-failed 0  -> MARGINAL   misspec-flagged seeds: 0   [30s this call]
[zonal] U0=0.5|rich|h                seeds=2  identified 1  not-id 1  opt-failed 0  -> MARGINAL   misspec-flagged seeds: 0   [33s this call]
[zonal] U0=1.0|rich|h                seeds=2  identified 2  not-id 0  opt-failed 0  -> IDENTIFIED   misspec-flagged seeds: 0   [31s this call]
[zonal] U0=0.15|sparse4|h            seeds=2  identified 0  not-id 2  opt-failed 0  -> NOT_IDENTIFIED   misspec-flagged seeds: 0   [33s this call]
[zonal] U0=0.5|sparse4|h             seeds=2  identified 1  not-id 1  opt-failed 0  -> MARGINAL   misspec-flagged seeds: 0   [31s this call]
[zonal] U0=1.0|sparse4|h             seeds=2  identified 2  not-id 0  opt-failed 0  -> IDENTIFIED   misspec-flagged seeds: 0   [32s this call]
[zonal] U0=0.15|rich|huv             seeds=2  identified 2  not-id 0  opt-failed 0  -> IDENTIFIED   misspec-flagged seeds: 0   [30s this call]
[

## Cell 8 — E5: misspecification
No pass/fail. Zonal-4 and free field are both fitted to truths that are *not* in the zonal class.

In [9]:
# ======================= E5: misspecification (no pass/fail; honest reporting) =======================
E5_T = ["curved", "three_zone", "gradient"]
done5 = {(r["regime"], r["truth"]) for r in RESULTS["E5"]}
t0 = time.time()
if RUN["E5"] and 1.0 in CFG["U0_LIST"]:
    for reg in REGIMES:
        for tname in E5_T:
            if (reg, tname) in done5: continue
            T = TRUTHS[tname]
            chi2_truth, zon = [], []
            for ns in range(CFG["N_SEEDS_E5"]):
                pbz = build_problem(1.0, reg, "h", CFG["P_OBS"], noise_seed=ns, cf_truth=T)
                chi2_truth.append(float(pbz["chi2"](jnp.array(T))))
                best = None
                for k in range(CFG["N_STARTS"]):
                    r = np.random.default_rng(100 + k + 17 * ns)
                    p0 = np.array([np.log(2.5e-3) + r.normal(0, 0.4), np.log(2.5e-3) + r.normal(0, 0.4), r.uniform(0.3, 0.7), r.uniform(-0.2, 0.2)])
                    s = fit_zonal(pbz, p0)
                    if best is None or s["chi2"] < best["chi2"]: best = s
                cfz = cf_zonal(np, np.array([np.log(best["cf_low"]), np.log(best["cf_high"]), best["b0"], best["s"]]))
                zon.append(dict(noise_seed=ns, chi2=best["chi2"], **field_metrics(cfz, T), cf_low=best["cf_low"], cf_high=best["cf_high"], b0=best["b0"], s=best["s"]))
            pb0 = build_problem(1.0, reg, "h", CFG["P_OBS"], noise_seed=0, cf_truth=T)
            rows, dp, orc, _ = fit_free_path(pb0, CFG["L_E2"], T)
            RESULTS["E5"].append(dict(regime=reg, truth=tname, chi2_at_truth=chi2_truth, zonal=zon, free_rows=rows, free_dp=dp, free_oracle=orc))
            save_results()
            print(f"[E5] {reg:8s} truth={tname:10s} [{time.time()-t0:.0f}s]")
print("\n=== E5 summary (U0=1, h-only). skill: 1 = perfect, 0 = no better than a constant, <0 = worse. chi2 ~ 1 means data are explained at noise level ===")
print(f"{'regime':8s} {'truth':10s} | {'zonal skill':>11} {'zonal r':>8} {'zonal chi2':>10} | {'free(DP) skill':>14} {'free r':>7} {'free chi2':>9} | {'chi2@truth':>10}")
for r in RESULTS["E5"]:
    z = r["zonal"]; f = r["free_rows"][r["free_dp"]]
    print(f"{r['regime']:8s} {r['truth']:10s} | {np.mean([a['skill'] for a in z]):11.2f} {np.mean([a['r'] for a in z]):8.2f} {np.mean([a['chi2'] for a in z]):10.2f} | "
          f"{f['skill']:14.2f} {f['r']:7.2f} {f['chi2_data']:9.2f} | {np.mean(r['chi2_at_truth']):10.2f}")
save_results()
print("\nSaved ->", OUT_PATH, "| pre-registration hash:", PREREG_HASH)


[E5] rich     truth=curved     [37s]
[E5] rich     truth=three_zone [75s]
[E5] rich     truth=gradient   [111s]
[E5] sparse4  truth=curved     [149s]
[E5] sparse4  truth=three_zone [183s]
[E5] sparse4  truth=gradient   [220s]

=== E5 summary (U0=1, h-only). skill: 1 = perfect, 0 = no better than a constant, <0 = worse. chi2 ~ 1 means data are explained at noise level ===
regime   truth      | zonal skill  zonal r zonal chi2 | free(DP) skill  free r free chi2 | chi2@truth
rich     curved     |        0.60     0.95       1.00 |           0.13    0.88      1.01 |       1.00
rich     three_zone |        0.43     0.87       0.99 |           0.13    0.87      1.00 |       1.00
rich     gradient   |        0.37     0.89       1.00 |           0.11    0.88      1.00 |       1.00
sparse4  curved     |        0.72     0.96       0.88 |           0.61    0.93      0.88 |       0.88
sparse4  three_zone |        0.42     0.82       1.02 |           0.65    0.94      0.90 |       0.88
sparse4  gradi